# **Decision  Layer and Business Actions**

## **I. Import Libraries**

In [ ]:
import pandas as pd
import numpy as np

## **II. Decision Layer Overview**

This notebook represents the final stage of the business-driven demand forecasting system. At this stage, forecast outputs are no longer treated as numerical predictions to be optimized, but as decision signals used to support operational prioritization and planning.

The objective of the decision layer is to translate validated forecast signals into:
- Risk-aware indicators
- Ranked priorities
- Actionable decision rules

This separation is intentional. Modeling focuses on generating reliable and stable signals, while decision logic focuses on 
how those signals are interpreted and acted upon within operational constraints.

By design, this layer emphasizes interpretability, controllability, and business alignment 
over algorithmic complexity.


## **III. Decision Philosophy & Design Principles**

Several principles guide the design of the decision layer:

1. Forecasts are directional signals, not absolute truths  
   The system prioritizes relative risk and ranking rather than exact forecast accuracy.

2. Decisions should be explainable to non-technical stakeholders  
   All decision rules are rule-based and transparent by design.

3. Actionability is more important than precision  
   A slightly imperfect but interpretable signal is preferred over a highly complex model 
   that is difficult to operationalize.

4. Human judgment remains part of the loop  
   This system is designed to support decision-making, not to replace managerial authority.


## **IV. Decision Unit and Granularity**

All decisions in this system are evaluated at the Store–Department–Time level. This granularity is intentionally selected because:
- It aligns with how operational decisions are typically executed in retail settings
- It balances signal stability with actionability
- It avoids excessive noise commonly observed at lower granularity levels (e.g., SKU-level)

The decision layer does not attempt SKU-level execution logic. Such execution-level decisions are assumed to be handled downstream by inventory planners or store-level operational teams.


## **V. Risk Signal Definition**

This section defines the core risk signals used in the decision layer.

The objective is not to predict operational outcomes directly, but to identify store–department combinations that exhibit systematic forecast bias and therefore require prioritized attention.

Only two primary risk signals are defined to maintain interpretability and operational focus:
- Under-Forecast Risk (stockout proxy)
- Over-Forecast Risk (overstock proxy)


### 5.1 Build Risk Signal

This step constructs the core risk signals used in the decision layer.

Rather than evaluating forecast accuracy in isolation, the focus here is on understanding **systematic forecast behavior**—specifically whether the model tends to **under-forecast** or **over-forecast** demand over time. These directional patterns serve as practical proxies for operational risk, such as potential stockouts or overstock situations.

Key design choices in this step include:
- Using **residuals from the validated test split** to ensure signals reflect real out-of-sample behavior
- Converting raw residuals into **directional indicators** (under-forecast vs over-forecast)
- Normalizing error magnitude using **absolute percentage error (APE)** with safeguards for low-volume periods

The resulting signals are intentionally simple, interpretable, and stable—designed to support downstream prioritization rather than precise error diagnosis.

In [58]:
# Load residuals table from modeling stage
df = pd.read_parquet("artifacts/forecast_residuals_val_test.parquet")

# Standardize names first
df = df.rename(columns={"Store": "store", "Dept": "dept", "Date": "date"}).copy()

# Focus on TEST split
df = df[df["split"] == "test"].copy()

# Use residual from modeling as truth:
# residual = y_true - y_pred  =>  error = y_pred - y_true = -residual
df["error"] = -df["residual"]
df["abs_error"] = df["error"].abs()

eps = 1e-6
df["ape"] = df["abs_error"] / np.maximum(df["y_true"], eps)

# Directional flags (based on error = y_pred - y_true)
df["under_flag"] = (df["error"] < 0).astype(int)  # y_pred < y_true
df["over_flag"]  = (df["error"] > 0).astype(int)  # y_pred > y_true

In [59]:
df["date"] = pd.to_datetime(df["date"])
for c in ["y_true", "y_pred", "residual"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

assert df["y_true"].notna().all() and df["y_pred"].notna().all()

### 5.2 Aggregate to Decision Unit (Store-Dept)

In this step, individual time-level risk signals are aggregated to the **store–department** level, which serves as the primary decision unit for this system.

This aggregation is intentional. Operational planning decisions—such as inventory review, replenishment adjustment, or promotional alignment—are typically executed at this level, where signals are more stable and actionable than at finer granularities (e.g., SKU-level).

For each store–department pair, the system summarizes:
- The **frequency of under-forecasting and over-forecasting** over time
- The **average error magnitude**, as a measure of sustained forecast deviation
- The **number of observed periods**, ensuring decisions are supported by sufficient historical evidence

By consolidating time-level behavior into a single decision unit, the system produces risk indicators that are easier to interpret, compare, and prioritize within real planning workflows.

In [60]:
group_cols = ["store", "dept"]

df_risk = (
    df.groupby(group_cols)
      .agg(
          under_rate=("under_flag", "mean"),
          over_rate=("over_flag", "mean"),
          avg_ape=("ape", "mean"),
          mean_error=("error", "mean"),
          n_periods=("error", "count"),
      )
      .reset_index()
)

df_risk.sort_values("avg_ape", ascending=False).head(10)

,store,dept,under_rate,over_rate,avg_ape,mean_error,n_periods
2457,36,72,0.000000,1.000000,1.038401e+09,1038.401044,1
810,12,49,0.000000,1.000000,6.351254e+08,635.125367,1
2836,42,71,0.000000,1.000000,2.779930e+08,277.992989,1
2374,35,19,1.000000,0.000000,1.909347e+08,-190.934665,1
1263,18,96,0.333333,0.666667,1.620040e+08,126.494044,3
1523,22,47,1.000000,0.000000,1.497427e+08,-149.742708,1
1356,20,19,0.000000,1.000000,1.489651e+08,148.965063,2
2030,29,80,0.333333,0.666667,9.200369e+07,122.726853,3
2828,42,44,1.000000,0.000000,8.038214e+07,-80.382139,1
2411,35,80,0.333333,0.666667,5.352100e+07,48.378393,3


**Interpreting Aggregated Risk Signals**

This table summarizes time-level forecast behavior into store–department–level risk indicators, which serve as the core input for downstream prioritization and action mapping.

Several important patterns can be observed:
- Extreme average APE values are often associated with low observation counts, indicating unstable or one-off deviations rather than persistent risk. This reinforces the need for minimum-period eligibility filters before decisions are escalated.
- Under-rate and over-rate provide directional context, helping distinguish whether forecast bias is skewed toward potential stockout risk (under-forecasting) or overstock risk (over-forecasting).
- Mean error complements magnitude-based metrics, offering insight into whether forecast deviations are systematically biased in one direction over time.

By aggregating signals at the store–department level, the system intentionally shifts focus away from isolated forecast errors and toward sustained behavioral patterns that are more actionable for operational planning.

These aggregated indicators are not used as decisions on their own, but as structured inputs for risk scoring, ranking, and tier-based action mapping in the subsequent steps.

## **VI. Risk Scoring and Priority Ranking**

Before ranking store–department priorities, we apply simple guardrails to ensure the risk signals are decision-safe. Percentage-based errors (like APE) can explode when actual demand is near zero, which may dominate rankings and distort operational focus. To prevent this, we (1) cap APE at a reasonable upper bound, and (2) require a minimum number of periods before a store–department can be prioritized.

In [61]:
MIN_PERIODS = 3
EPS = 1e-6

group_cols = ["store", "dept"]

df_risk_wape = (
    df.groupby(group_cols)
      .agg(
          under_rate=("under_flag", "mean"),
          over_rate=("over_flag", "mean"),
          mean_error=("error", "mean"),
          n_periods=("error", "count"),
          sum_abs_error=("abs_error", "sum"),
          sum_abs_actual=("y_true", lambda s: np.abs(s).sum()),  # <-- robust denominator
      )
      .reset_index()
)

df_risk_wape["wape"] = df_risk_wape["sum_abs_error"] / np.maximum(df_risk_wape["sum_abs_actual"], EPS)

df_risk_wape["eligible"] = df_risk_wape["n_periods"] >= MIN_PERIODS

# (under_rate + over_rate) ~= 1, but keep as clarity
df_risk_wape["risk_score"] = (df_risk_wape["under_rate"] + df_risk_wape["over_rate"]) * df_risk_wape["wape"]

df_priority = (
    df_risk_wape[df_risk_wape["eligible"]]
      .assign(risk_rank=lambda x: x["risk_score"].rank(ascending=False, method="dense"))
      .sort_values("risk_rank")
      .reset_index(drop=True)
)

df_priority[["store","dept","n_periods","sum_abs_actual","sum_abs_error","wape","risk_score","risk_rank"]].head(20)


,store,dept,n_periods,sum_abs_actual,sum_abs_error,wape,risk_score,risk_rank
0,18,96,3,15.45,486.012030,31.457089,31.457089,1.0
1,35,49,3,14.00,356.471395,25.462242,25.462242,2.0
2,29,94,3,10.88,232.437003,21.363695,21.363695,3.0
3,8,99,3,50.06,817.400563,16.328417,16.328417,4.0
4,5,19,3,82.20,1066.712559,12.977038,12.977038,5.0
5,7,59,3,21.64,222.290893,10.272222,10.272222,6.0
6,29,19,3,25.00,229.622890,9.184916,9.184916,7.0
7,38,44,3,22.41,204.065325,9.105994,9.105994,8.0
8,30,9,3,26.14,209.651496,8.020333,8.020333,9.0
9,37,72,3,311.22,2396.880196,7.701562,7.701562,10.0


In [62]:
df_priority[["store","dept","n_periods","sum_abs_actual","sum_abs_error","wape","risk_score","risk_rank"]].head(20)


,store,dept,n_periods,sum_abs_actual,sum_abs_error,wape,risk_score,risk_rank
0,18,96,3,15.45,486.012030,31.457089,31.457089,1.0
1,35,49,3,14.00,356.471395,25.462242,25.462242,2.0
2,29,94,3,10.88,232.437003,21.363695,21.363695,3.0
3,8,99,3,50.06,817.400563,16.328417,16.328417,4.0
4,5,19,3,82.20,1066.712559,12.977038,12.977038,5.0
5,7,59,3,21.64,222.290893,10.272222,10.272222,6.0
6,29,19,3,25.00,229.622890,9.184916,9.184916,7.0
7,38,44,3,22.41,204.065325,9.105994,9.105994,8.0
8,30,9,3,26.14,209.651496,8.020333,8.020333,9.0
9,37,72,3,311.22,2396.880196,7.701562,7.701562,10.0


**Interpretation of Risk Scores and Priority Ranking**

The table above represents the final output of the decision layer: a ranked list of store–department units prioritized by forecast risk severity.

Each unit is scored using a composite risk score that combines:
- **Error magnitude**, measured via WAPE to capture sustained deviation over time
- **Directional bias**, reflected through under- and over-forecast rates
- **Evidence sufficiency**, enforced through a minimum number of observed periods

As a result, higher-ranked units are not simply those with large absolute errors, but those that exhibit **consistent and material forecast misalignment** relative to their actual demand.

---

**Key Observations**

Several patterns emerge from the top-ranked entries:
- High-risk units tend to combine **large cumulative forecast errors** with **relatively low total actual volume**, amplifying operational impact.
- Units with the same number of observed periods can still rank very differently, indicating that **error magnitude and persistence**, rather than sample size alone, drive prioritization.
- The ranking remains stable and interpretable due to the use of WAPE and minimum-period guardrails, preventing extreme values from dominating the list.

---

**Operational Implications**

This ranking is designed to support **focused managerial attention**, not automated execution.  
Store–department units appearing at the top of the list should be considered candidates for:
- Forecast review and recalibration
- Inventory policy adjustment
- Promotional or replenishment alignment checks

Importantly, this list does not prescribe SKU-level actions. Instead, it serves as a **risk-aware prioritization tool**, enabling planners and managers to allocate time and resources where forecast risk is most concentrated. Human judgment remains an explicit part of the decision loop.


## **VII. Action Mapping**

Risk scores are translated into decision tiers to support operational execution. Each tier is directly associated with a predefined operational action, allowing the output to be consumed quickly by planners and non-technical stakeholders. The mapping is intentionally kept simple to prioritize clarity, usability, and decision speed.


### 7.1 Risk Tiering

In this step, continuous risk scores are translated into discrete risk tiers (HIGH, MEDIUM, LOW) using quantile-based thresholds. This approach ensures robustness across varying data scales and distributions, while keeping the tiering logic intuitive and easily explainable to non-technical stakeholders.


In [68]:
# --- Safety: ensure numeric types (prevents lexicographic sorting bugs) ---
df_priority["risk_score"] = pd.to_numeric(df_priority["risk_score"], errors="coerce")
df_priority["risk_rank"]  = pd.to_numeric(df_priority["risk_rank"],  errors="coerce")

# Quantile-based tiering (compute on eligible population only; df_priority already eligible)
high_th = df_priority["risk_score"].quantile(0.90)
med_th  = df_priority["risk_score"].quantile(0.70)

# Vectorized tier assignment (faster + cleaner than apply)
df_priority["risk_tier"] = np.select(
    [
        df_priority["risk_score"] >= high_th,
        df_priority["risk_score"] >= med_th
    ],
    ["HIGH", "MEDIUM"],
    default="LOW"
)


### 7.2 Action Rules

Each risk tier is mapped to a predefined operational action. The rules are intentionally simple to support fast decision-making and ensure practical usability within real-world planning workflows.


In [69]:
ACTION_MAP = {
    "HIGH":   "Immediate review required: validate demand drivers, promotion effects, and stock readiness",
    "MEDIUM": "Monitor closely and reassess in the next planning cycle",
    "LOW":    "No immediate action required"
}

df_priority["recommended_action"] = df_priority["risk_tier"].map(ACTION_MAP)


### 7.3 Final Decision Output

This section presents the final decision-ready output by combining risk ranking, tier classification, and actionable recommendations. The result is a prioritized list of store–department units that can be directly consumed by planners and operational teams.


In [70]:
# Sort safely (numeric) and build final decision output
df_decision = (
    df_priority
    .sort_values(["risk_rank", "risk_score"], ascending=[True, False])
    .reset_index(drop=True)
)

df_decision.head(10)


,store,dept,under_rate,over_rate,mean_error,n_periods,sum_abs_error,sum_abs_actual,wape,eligible,risk_score,risk_rank,risk_tier,recommended_action
0,18,96,0.333333,0.666667,126.494044,3,486.012030,15.45,31.457089,True,31.457089,1.0,HIGH,Immediate review required: validate demand dri...
1,35,49,0.000000,1.000000,118.823798,3,356.471395,14.00,25.462242,True,25.462242,2.0,HIGH,Immediate review required: validate demand dri...
2,29,94,0.333333,0.666667,25.101909,3,232.437003,10.88,21.363695,True,21.363695,3.0,HIGH,Immediate review required: validate demand dri...
3,8,99,0.000000,1.000000,272.466854,3,817.400563,50.06,16.328417,True,16.328417,4.0,HIGH,Immediate review required: validate demand dri...
4,5,19,0.666667,0.333333,-302.765882,3,1066.712559,82.20,12.977038,True,12.977038,5.0,HIGH,Immediate review required: validate demand dri...
5,7,59,0.333333,0.666667,53.196977,3,222.290893,21.64,10.272222,True,10.272222,6.0,HIGH,Immediate review required: validate demand dri...
6,29,19,0.333333,0.666667,70.799259,3,229.622890,25.00,9.184916,True,9.184916,7.0,HIGH,Immediate review required: validate demand dri...
7,38,44,1.000000,0.000000,-68.021775,3,204.065325,22.41,9.105994,True,9.105994,8.0,HIGH,Immediate review required: validate demand dri...
8,30,9,0.000000,1.000000,69.883832,3,209.651496,26.14,8.020333,True,8.020333,9.0,HIGH,Immediate review required: validate demand dri...
9,37,72,0.000000,1.000000,798.960065,3,2396.880196,311.22,7.701562,True,7.701562,10.0,HIGH,Immediate review required: validate demand dri...


**Interpretation of Final Decision Output**

The table above represents the final, decision-ready output of the system. Each row corresponds to a store–department unit, enriched with risk metrics, priority ranking, risk tier classification, and a recommended managerial action. This output is intentionally structured to be **directly consumable by planners and operational teams**, without requiring interpretation of model internals or statistical artifacts.

---

**How to Read This Table**

- **Risk Rank** defines *relative priority* across all eligible store–department units.
- **Risk Tier** translates numerical risk scores into intuitive severity bands (e.g., HIGH), enabling quick scanning and escalation.
- **Recommended Action** provides a standardized response guideline aligned with the identified risk level.

Units appearing at the top of the table represent locations where forecast deviations are both **material and persistent**, and therefore warrant immediate attention.

---

**Operational Usage**

This table is designed to support structured planning workflows, such as:
- Weekly or monthly forecast review meetings
- Inventory and replenishment alignment checks
- Cross-functional escalation between demand planning and store operations

Importantly, the recommended actions are **guidance, not automation**. They are intended to focus attention and support decision-making, while allowing planners and managers to apply contextual judgment based on local knowledge and constraints.

---

**Decision Boundary**

The system deliberately stops at the store–department level. SKU-level execution, quantity adjustments, and final operational decisions remain the responsibility of downstream planning teams. This preserves accountability, interpretability, and human oversight within the decision loop.


## **VIII. Executive Summary**

### 8.1 Content Recap

Demand forecasting often fails not because predictions are inaccurate, but because they are difficult to translate into concrete operational decisions. In real planning environments, teams need clarity on where to act, how urgent the situation is, and what should be done next—not another layer of technical outputs.

This system was built to bridge that gap. Instead of stopping at forecast results, it focuses on converting forecast performance signals into a structured, decision-ready view that planners and operations teams can immediately work with.

### 8.2 What This System Delivers

The system produces a prioritized list of store–department units, each enriched with:
- a relative risk ranking,
- a clear risk tier (HIGH / MEDIUM / LOW),
- and a predefined operational recommendation.

The output is not a model artifact, but a decision artifact—designed to be read, interpreted, and acted upon without requiring technical context.

### 8.3 How It Supports Business Decisions

With this structure, planners can quickly identify:
- which store–department combinations require immediate attention,
- which ones should be monitored in the next planning cycle,
- and which areas are currently stable.

This allows teams to focus discussions on concrete actions such as validating demand drivers, checking promotion effects, or reviewing stock readiness—turning forecast insights into practical planning decisions rather than abstract analysis.

### 8.4 Limitations & Next Iteration

- The current action rules are intentionally simple to ensure clarity and usability for non-technical stakeholders.
- Future iterations can incorporate business-specific thresholds or cost-based prioritization to further refine decision impact.
- Integration with downstream planning or replenishment systems would allow recommendations to flow directly into execution.